In [2]:
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.optimize import curve_fit
import scipy.constants as cs
from scipy.interpolate import interp1d

#### Note
The following specs were taken from the ThorLabs PDA55 manual, which has a working area of 3.6 x 3.6 mm

In [3]:
noise = 0.33e-3 # V, noise level in the measurement
qe = 0.5
r = 50 # Ohms, resistance of the photodetector
transimpedance_gain = 1.5e4 # V/A, gain of the transimpedance amplifier
c = cs.e
current_noise = noise / transimpedance_gain # Amps 

In [10]:
incident_wavelength = 637e-9 # m https://www.nature.com/articles/nphys318
data = np.genfromtxt("plot-data.csv", delimiter=",", skip_header=1)
data = np.unique(data, axis=0) 

In [5]:
photons_required = current_noise / (qe * c) # photons/s
print(f"Number of incident photons required to produce a signal equal to the noise level: {photons_required:.2e} photons/s")

Number of incident photons required to produce a signal equal to the noise level: 2.75e+11 photons/s


### Number of incident photons per second (to create a current equal to the RMS noise) $P_s$:
$P_s = I_{noise} * \frac{1}{q_e*\gamma}$

where $I_{noise}$ is the current noise at the 0dB setting in amps, $q_e$ is the charge of an electron in couloumbs, and $\gamma$ is the quantum efficiency of the pda55, obtained from this paper:https://chapmanlabs.gatech.edu/papers/kevin-fortier.pdf

In [6]:
photon_energy = cs.h * cs.c / incident_wavelength # J
power_required = photons_required * photon_energy # W
print(f"Power required to produce a signal equal to the noise level: {power_required:.2e} W")

Power required to produce a signal equal to the noise level: 8.56e-08 W


In [15]:
wavelengths = data[:, 0] # m
responsivities = data[:, 1] # A/W
responsivity_interp = interp1d(wavelengths, responsivities, kind="cubic")
responsivity_at_incident = responsivity_interp(incident_wavelength) # A/W
power_required_interp = current_noise / responsivity_at_incident # W
print(f"Power required for a signal-to-noise ratio of 1 at {incident_wavelength*1e9:.0f} nm: {power_required_interp:.4e} W")

Power required for a signal-to-noise ratio of 1 at 637 nm: 5.0070e-08 W


In [16]:
# Strategy using https://media.thorlabs.com/globalassets/items/p/pd/pda/pda55/2058-d02.pdf?v=0116020238's datasheet which provided an equation to convert light to voltage, then using the voltage noise. 
volts_per_watt = transimpedance_gain * responsivity_at_incident # V/W
power_required_snr1 = noise / volts_per_watt # W
print(f"Power required for a signal-to-noise ratio of 1 at {incident_wavelength*1e9:.0f} nm: {power_required_snr1:.4e} W")

Power required for a signal-to-noise ratio of 1 at 637 nm: 5.0070e-08 W
